# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [58]:
# importar librerías

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [59]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [60]:
# explorar datasets
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [61]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [62]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [63]:
# tu código aquí
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


In [64]:
# Revisar variables numéricas (sin negativos o ceros inválidos) 
orders[['cantidad','precio_unitario','monto_descuento','monto_total']].describe()
orders = orders[orders['cantidad'] >= 0].copy()

In [65]:

# cantidad de nulos para orders
print(orders.isna().sum())
print(orders.isna().mean())

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
dtype: int64
id_pedido             0.000000
id_usuario            0.000000
fecha_hora_pedido     0.000000
pais                  0.011818
dispositivo           0.000799
fuente_referencia     0.001198
nombre_producto       0.001198
categoria_producto    0.001198
cantidad              0.000000
precio_unitario       0.000000
monto_descuento       0.000000
monto_total           0.000000
dtype: float64


In [66]:
orders['monto_descuento'] = orders['monto_descuento'].fillna(0)

In [67]:

orders_variables_categoricas = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
orders = orders.dropna(subset=orders_variables_categoricas)

In [68]:
orders = orders.dropna(subset=['cantidad', 'precio_unitario'])

In [69]:
orders = orders[orders['cantidad'] >= 0].copy()

In [70]:

# Verificar consistencia de montos
orders['monto_calculado'] = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

orders['diferencia'] = (orders['monto_total'] - orders['monto_calculado']).abs()

# Ver el resumen estadístico de las diferencias
print(orders['diferencia'].describe())

count    2.470000e+04
mean     2.495951e-03
std      4.327875e-03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.842171e-14
max      1.000000e-02
Name: diferencia, dtype: float64


In [71]:
orders = orders.drop(columns=['monto_calculado', 'diferencia'])
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24700 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24700 non-null  object        
 1   id_usuario          24700 non-null  object        
 2   fecha_hora_pedido   24700 non-null  datetime64[ns]
 3   pais                24700 non-null  object        
 4   dispositivo         24700 non-null  object        
 5   fuente_referencia   24700 non-null  object        
 6   nombre_producto     24700 non-null  object        
 7   categoria_producto  24700 non-null  object        
 8   cantidad            24700 non-null  float64       
 9   precio_unitario     24700 non-null  float64       
 10  monto_descuento     24700 non-null  float64       
 11  monto_total         24700 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.4+ MB


In [72]:
orders[orders.duplicated(keep=False)].sort_values('id_pedido')

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25048,order_8326,user_6177,2025-06-14,Argentina,mobile,organic,Vacuum-Pro-Black,Hogar,1.0,457.53,0.0,457.53
25043,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.71
8414,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.71
25056,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16


In [73]:
orders = orders.drop_duplicates()
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24600 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24600 non-null  object        
 1   id_usuario          24600 non-null  object        
 2   fecha_hora_pedido   24600 non-null  datetime64[ns]
 3   pais                24600 non-null  object        
 4   dispositivo         24600 non-null  object        
 5   fuente_referencia   24600 non-null  object        
 6   nombre_producto     24600 non-null  object        
 7   categoria_producto  24600 non-null  object        
 8   cantidad            24600 non-null  float64       
 9   precio_unitario     24600 non-null  float64       
 10  monto_descuento     24600 non-null  float64       
 11  monto_total         24600 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.4+ MB


In [74]:
for col in orders_variables_categoricas:
    orders[col] = orders[col].str.strip().str.capitalize()

In [75]:
orders['pais'].value_counts()

Mexico       8305
Colombia     8273
Argentina    8022
Name: pais, dtype: int64

In [76]:
orders['dispositivo'].value_counts()

Desktop    12524
Mobile     12076
Name: dispositivo, dtype: int64

In [77]:
orders['fuente_referencia'].value_counts()

Social         8268
Organic        8190
Paid_search    8142
Name: fuente_referencia, dtype: int64

In [78]:
orders['nombre_producto'].value_counts()

Jacket-winter-m         4122
Vacuum-pro-black        4114
Blender-xl-red          4114
Sneakers-urban-42       4072
Laptop-gaming-16gb      2752
Tablet-standard-64gb    2729
Phone-pro-128gb         2697
Name: nombre_producto, dtype: int64

In [79]:
orders['categoria_producto'].value_counts()

Hogar          8228
Moda           8194
Electronica    8178
Name: categoria_producto, dtype: int64

In [80]:
#Limpieza tabla catalog
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [81]:
catalog_variables_categoricas=['nombre_producto','categoria_producto','proveedor']
for col in catalog_variables_categoricas:
    catalog[col] = catalog[col].str.strip().str.capitalize()
    print(catalog[col].value_counts())
    print()

Jacket-winter-m         1
Phone-pro-128gb         1
Vacuum-pro-black        1
Tablet-standard-64gb    1
Sneakers-urban-42       1
Blender-xl-red          1
Laptop-gaming-16gb      1
Name: nombre_producto, dtype: int64

Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

King ltd                   1
Long-reid                  1
Rivera, carr and finley    1
Fuller, pena and myers     1
Greene-smith               1
Bowers llc                 1
Mcmillan-rhodes            1
Name: proveedor, dtype: int64



In [82]:
#Limpiezaz marketing
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [83]:
marketing_variables_categoricas=['pais','id_campaña','canal']
for col in marketing_variables_categoricas:
    marketing[col] = marketing[col].str.strip().str.capitalize()
    print(marketing[col].value_counts())
    print()

Argentina    540
Colombia     540
Mexico       540
Name: pais, dtype: int64

Social_mexico            180
Organic_colombia         180
Paid_search_mexico       180
Organic_mexico           180
Paid_search_colombia     180
Paid_search_argentina    180
Social_argentina         180
Social_colombia          180
Organic_argentina        180
Name: id_campaña, dtype: int64

Paid_search    507
Social         506
Organic        506
Name: canal, dtype: int64



In [84]:
marketing.duplicated().sum()

0

In [85]:
# cantidad de nulos para marketing
print(marketing.isna().sum())
print(marketing.isna().mean())

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64
fecha         0.000000
pais          0.000000
id_campaña    0.000000
canal         0.062346
gasto         0.000000
dtype: float64


In [86]:
marketing[['canal_extraido', 'pais_extraido']] = marketing['id_campaña'].str.rsplit('_', n=1, expand=True)

In [87]:
marketing[['id_campaña', 'canal_extraido', 'pais_extraido']].head(10)

,id_campaña,canal_extraido,pais_extraido
0,Organic_mexico,Organic,mexico
1,Paid_search_mexico,Paid_search,mexico
2,Social_mexico,Social,mexico
3,Organic_colombia,Organic,colombia
4,Paid_search_colombia,Paid_search,colombia
5,Social_colombia,Social,colombia
6,Organic_argentina,Organic,argentina
7,Paid_search_argentina,Paid_search,argentina
8,Social_argentina,Social,argentina
9,Organic_mexico,Organic,mexico


In [88]:
marketing['canal'] = marketing['canal'].fillna(marketing['canal_extraido'])

In [89]:
marketing['canal'].value_counts()

Social         540
Organic        540
Paid_search    540
Name: canal, dtype: int64

In [90]:
marketing.isna().sum()

fecha             0
pais              0
id_campaña        0
canal             0
gasto             0
canal_extraido    0
pais_extraido     0
dtype: int64

In [91]:
marketing = marketing.drop(columns=['canal_extraido', 'pais_extraido'])

In [92]:

orders['pais'].value_counts()

Mexico       8305
Colombia     8273
Argentina    8022
Name: pais, dtype: int64

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [93]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
  51966981.56
- ¿Cuál es el costo total?
  43124069.01
- ¿Cuánto se ha invertido en marketing?
  2871843.53
- ¿El negocio es rentable? (calcular profit)
    8842912.54

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
   2083.17
- ¿Cuál es la cantidad promedio de productos por orden?
   7.11
- ¿Cuál es el producto más vendido?
  Laptop-gaming-16gb      144198.0
- ¿Cuánto se ha gastado en marketing por canal?
   Social         976818.37

In [57]:
# tu código aquí
orders_revenue = orders['monto_total'].sum()

print("Ingreso Total",orders_revenue)

Ingreso Total 51966981.56


In [58]:
df_combinado = orders.merge(catalog, on='nombre_producto', how='left')
df_combinado['costo_total'] = df_combinado["costo_unitario"]*df_combinado['cantidad']
costo_total = df_combinado['costo_total'].sum()
print('Costo Total',costo_total)

Costo Total 43124069.010000005


In [59]:
inversion_marketing=marketing['gasto'].sum()
print('Inversion en Marketing',inversion_marketing)

Inversion en Marketing 2871843.53


In [60]:
profit= orders_revenue - costo_total
print('Profit:',profit)

Profit: 8842912.549999997


In [61]:
ticket_promedio = orders['monto_total'].mean()
print('Ticket Promedio:', ticket_promedio)

Ticket Promedio: 2083.1789288863947


In [62]:
productos_promedio = orders['cantidad'].mean()
print('Cantidad Promedio de Productos por Orden',productos_promedio)

Cantidad Promedio de Productos por Orden 7.116451535316283


In [63]:
marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

canal
Social         976818.37
Organic        972650.96
Paid_search    922374.20
Name: gasto, dtype: float64

In [64]:
orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

nombre_producto
Laptop-gaming-16gb      144198.0
Vacuum-pro-black          6284.0
Blender-xl-red            6279.0
Jacket-winter-m           6256.0
Sneakers-urban-42         6172.0
Tablet-standard-64gb      4153.0
Phone-pro-128gb           4140.0
Desconocido                 45.0
Name: cantidad, dtype: float64

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [65]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [66]:
# PARTE 1: Totales del funnel
# =========================
# 	add_payment_info
#1	first_visit
#2	add_to_cart
#3	select_item
#4	begin_checkout
#5 add_payment info

query_events = '''
SELECT 
nombre_evento,
Count(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250


In [67]:
# PARTE 2: Conversiones
# ======================

query_totals = '''
WITH first_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
),
select_item AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
),

begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
),
events_counts AS(
SELECT
  COUNT(DISTINCT fv.id_usuario) AS usuarios_first_visit,
  COUNT(DISTINCT si.id_usuario) AS usuarios_select_item,
  COUNT(DISTINCT atc.id_usuario) AS usuarios_add_to_cart,
  COUNT(DISTINCT bc.id_usuario) AS usuarios_begin_checkout,
  COUNT(DISTINCT api.id_usuario) AS usuarios_add_payment_info
FROM first_visit AS fv
LEFT JOIN select_item si        ON fv.id_usuario = si.id_usuario 
LEFT JOIN add_to_cart atc         ON fv.id_usuario = atc.id_usuario
LEFT JOIN begin_checkout bc     ON fv.id_usuario = bc.id_usuario
LEFT JOIN add_payment_info api     ON fv.id_usuario = api.id_usuario
)
SELECT
usuarios_select_item * 100.0 / NULLIF(usuarios_first_visit,0) AS conversion_select_item,
usuarios_add_to_cart * 100.0 / NULLIF(usuarios_first_visit,0) AS conversion_add_to_cart,
usuarios_begin_checkout * 100.0 / NULLIF(usuarios_first_visit,0) AS conversion_begin_checkout,
usuarios_add_payment_info * 100.0 / NULLIF(usuarios_first_visit,0) AS conversion_add_payment_info
FROM events_counts
ORDER BY conversion_begin_checkout desc;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,conversion_select_item,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info
0,94.830682,95.420729,90.187276,78.091329


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [68]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM user_activity;
'''
users = pd.read_sql(query_users, con=engine)
users.head(10)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0
5,user_1,2025-01-21,14,0
6,user_1,2025-01-28,21,1
7,user_1,2025-02-04,28,0
8,user_2,2025-03-19,7,0
9,user_2,2025-03-26,14,1


In [69]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
-- 1) CTE para chorte:
WITH cohortes AS (
SELECT
id_usuario,
TO_CHAR(DATE_TRUNC('month', MIN(CAST(fecha_registro AS DATE))), 'YYYY-MM') AS cohort
FROM users
GROUP BY id_usuario
),
activity AS (
SELECT 
    user_activity.id_usuario,
    cohortes.cohort,
    user_activity.dias_despues_registro,
    user_activity.activo
FROM user_activity
LEFT JOIN cohortes ON cohortes.id_usuario = user_activity.id_usuario
)

SELECT cohort,
    COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 7 AND activo = 1 THEN id_usuario END) AS retention_d7_pct,
    COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 14 AND activo = 1 THEN id_usuario END) AS retention_d14_pct,
    COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 21 AND activo = 1 THEN id_usuario END) AS retention_d21_pct,
    COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 28 AND activo = 1 THEN id_usuario END) AS retention_d28_pct
FROM activity
GROUP BY cohort
ORDER BY cohort;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(10)

,cohort,retention_d7_pct,retention_d14_pct,retention_d21_pct,retention_d28_pct
0,2025-01,1381,1253,1027,671
1,2025-02,1255,1154,940,575
2,2025-03,1428,1306,1060,673
3,2025-04,1394,1261,1022,652
4,2025-05,1446,1321,1088,679


In [70]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
-- 1) CTE para chorte:
WITH cohortes AS (
SELECT
id_usuario,
TO_CHAR(DATE_TRUNC('month', MIN(CAST(fecha_registro AS DATE))), 'YYYY-MM') AS cohort
FROM users
GROUP BY id_usuario
),
activity AS (
SELECT 
    user_activity.id_usuario,
    cohortes.cohort,
    user_activity.dias_despues_registro,
    user_activity.activo
FROM user_activity
LEFT JOIN cohortes ON cohortes.id_usuario = user_activity.id_usuario
)

SELECT cohort,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 7 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d7_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 14 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d14_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 21 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d21_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 28 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d28_pct
FROM activity
GROUP BY cohort
ORDER BY cohort;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,cohort,retention_d7_pct,retention_d14_pct,retention_d21_pct,retention_d28_pct
0,2025-01,84.9,77.0,63.1,41.2
1,2025-02,86.9,79.9,65.1,39.8
2,2025-03,87.3,79.8,64.8,41.1
3,2025-04,86.8,78.5,63.6,40.6
4,2025-05,85.7,78.3,64.5,40.2


In [71]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
-- 1) CTE para chorte:
WITH cohortes AS (
SELECT
id_usuario,
TO_CHAR(DATE_TRUNC('month', MIN(CAST(fecha_registro AS DATE))), 'YYYY-MM') AS cohort
FROM users
GROUP BY id_usuario
),
activity AS (
SELECT 
    user_activity.id_usuario,
    cohortes.cohort,
    user_activity.dias_despues_registro,
    user_activity.activo
FROM user_activity
LEFT JOIN cohortes ON cohortes.id_usuario = user_activity.id_usuario
)

SELECT cohort,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 7 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d7_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 14 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d14_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 21 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d21_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN dias_despues_registro  >= 28 AND activo = 1 THEN id_usuario END) / NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d28_pct
FROM activity
GROUP BY cohort
ORDER BY cohort;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,cohort,retention_d7_pct,retention_d14_pct,retention_d21_pct,retention_d28_pct
0,2025-01,84.9,77.0,63.1,41.2
1,2025-02,86.9,79.9,65.1,39.8
2,2025-03,87.3,79.8,64.8,41.1
3,2025-04,86.8,78.5,63.6,40.6
4,2025-05,85.7,78.3,64.5,40.2


---


## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado*
*  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [72]:

# tu código aquí
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest

experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')




In [73]:
experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [74]:
experiment.head(10)

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12
5,exp_user_5,control,0,mobile,Mexico,206.70,2025-01-26
6,exp_user_6,control,0,desktop,Mexico,280.11,2025-04-15
7,exp_user_7,tratamiento,0,mobile,Argentina,28.47,2025-03-18
8,exp_user_8,tratamiento,0,desktop,Mexico,287.04,2025-02-05
9,exp_user_9,control,0,mobile,Argentina,258.91,2025-02-11


In [75]:
experiment['timestamp'] = pd.to_datetime(experiment['timestamp'], errors='coerce')

In [76]:
variables_numericas = ['duracion_sesion']
experiment[variables_numericas].describe()

,duracion_sesion
count,10000.000000
mean,159.862439
std,81.074410
min,20.010000
25%,89.470000
50%,159.725000
75%,229.745000
max,300.000000


In [77]:
variables_categoricas = ['variante','dispositivo','pais']
experiment[variables_categoricas].describe()

,variante,dispositivo,pais
count,10000,10000,10000
unique,2,2,3
top,tratamiento,desktop,Mexico
freq,5035,5042,3405


In [78]:

for col in variables_categoricas:
    print(f"{col}:", experiment[col].value_counts())
    print()

variante: tratamiento    5035
control        4965
Name: variante, dtype: int64

dispositivo: desktop    5042
mobile     4958
Name: dispositivo, dtype: int64

pais: Mexico       3405
Argentina    3317
Colombia     3278
Name: pais, dtype: int64



In [79]:
variables_binarias = ['convirtio']
experiment[variables_binarias].value_counts()

convirtio
0            8401
1            1599
dtype: int64

In [80]:
control = experiment[experiment['variante'] == 'control']['convirtio']
tratamiento = experiment[experiment['variante'] == 'tratamiento']['convirtio']

conversiones = [control.sum(), tratamiento.sum()]  
tamaños = [len(control), len(tratamiento)]

print(f"Cantidad de conversión control: {control.sum()}")
print(f"Cantidad de conversión tratamiento: {tratamiento.sum()}")

Cantidad de conversión control: 779
Cantidad de conversión tratamiento: 820


In [81]:
z_stat, p_value = proportions_ztest(conversiones, tamaños)

print(f"Estadístico z: {z_stat:.4f}")
print(f"Valor P: {p_value:.4f}")

Estadístico z: -0.8133
Valor P: 0.4161


In [82]:
alpha= 0.05
if p_value < alpha:
    print('Rechazamos la hipotesis nula, existe una diferencia')
else:
    print('No rechazamos la hipotesis nula, no existe evidencia suficiente de una diferencia')

No rechazamos la hipotesis nula, no existe evidencia suficiente de una diferencia


In [83]:
print(f"Tasa de conversión control: {control.mean():.4f}")
print(f"Tasa de conversión tratamiento: {tratamiento.mean():.4f}")
print(f"Tamaño control: {len(control)}, tratamiento: {len(tratamiento)}")

Tasa de conversión control: 0.1569
Tasa de conversión tratamiento: 0.1629
Tamaño control: 4965, tratamiento: 5035


El resultado no es estadísticamente significativo (p < 0.05) por lo que refleja que  no existe una diferencia entre el cambio de variantes 'control' y 'tratamiento'

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de one drive / google drive

https://drive.google.com/drive/folders/1l8E13vYSdMiCx1NTvXzB-U-95kvIdHKm?usp=sharing